In [ ]:
# --- paths come from human/config.py (auto-inserted by fix_notebooks.py) ---
import sys; sys.path.append('..')
from config import HUMAN_BASE


# Build SETIA Linked-Group (LG) File

**Goal**: Construct `GRN_BMMC_LG.txt` — the BMMC equivalent of yeast pipeline's `GRN_ssTFs_Sc_LG.txt`. This file encodes which TFs co-bind as **complexes** on each target gene, used by SETIA inference to constrain joint regulation.

**Yeast pipeline algorithm** (from `GRN_input_acquisition.py`):
1. For each TF→target edge, get the TF's ChIP-exo peak position in the target's NDR
2. Cluster co-binding TFs on each target by peak-position proximity (50 bp)
3. For each source TF, emit a partition string: which TFs share its complex on each target

**BMMC adaptation (Path B)**:
- ChIP-Atlas peaks have ~200-500 bp resolution (not ChIP-exo) → can't use peak-distance clustering
- Instead, use **STRING PPI** as the complex-membership signal: two TFs co-binding on a target are in the same complex iff they have a high-confidence STRING PPI
- Algorithm is **isomorphic** to yeast — only the source of co-binding signal changes

**Inputs** (from previous notebooks):
- `ppi_prior_output/BMMC_TF_TF_PPI_prior.json` — STRING TF-TF PPI edges
- `chip_prior_output/BMMC_TF_DNA_motif_chip_union.json` — ChIP+motif TF-target edges  
- `bmmc_gene_list.tsv` — 88-gene order (Column_order in yeast)

**Output**: `BMMC_LG.txt` — 88 × 88 partition string in yeast format

## Part A: Setup and load all inputs

In [ ]:
import json
import pandas as pd
from pathlib import Path
from collections import defaultdict

GENE_LIST_PATH = f"{HUMAN_BASE}/bmmc_gene_list.tsv"
CHIP_JSON_PATH = f"{HUMAN_BASE}/GTEx_v11/chip_prior_output/BMMC_TF_DNA_motif_chip_union.json"
PPI_JSON_PATH  = f"{HUMAN_BASE}/GTEx_v11/ppi_prior_output/BMMC_TF_TF_PPI_prior.json"
OUTPUT_DIR     = Path(f"{HUMAN_BASE}/GTEx_v11/lg_output");  OUTPUT_DIR.mkdir(exist_ok=True)

# Load gene list (Column_order in yeast)
gene_df = pd.read_csv(GENE_LIST_PATH, sep='\t', comment='#')
all_genes  = gene_df['gene_symbol'].tolist()
tf_genes   = gene_df[gene_df['category'] != 'marker']['gene_symbol'].tolist()
tf_set     = set(tf_genes)
Column_order = all_genes   # rename to match yeast variable name
n = len(Column_order)

print(f'Column_order length: {n} genes  (56 TFs + 32 markers)')
print(f'First 10: {Column_order[:10]}')

In [ ]:
# Load ChIP-DNA prior (TF-target binding)
with open(CHIP_JSON_PATH) as f:
    chip_data = json.load(f)

TF_DNA = defaultdict(list)
for e in chip_data['edges']:
    TF_DNA[e['source']].append(e['target'])
TF_DNA = dict(TF_DNA)

n_edges = sum(len(v) for v in TF_DNA.values())
print(f'ChIP-DNA prior:  {len(TF_DNA)} sources, {n_edges} TF→target edges')
print(f'Sample TF (GATA1) targets: {TF_DNA.get("GATA1", [])[:8]}')

In [ ]:
# Load PPI prior (TF-TF interactions, symmetric)
with open(PPI_JSON_PATH) as f:
    ppi_data = json.load(f)

PPI = set()   # set of (TF_a, TF_b) tuples, both directions stored
for e in ppi_data['edges']:
    PPI.add((e['source'], e['target']))

print(f'PPI prior: {len(PPI)} directed entries ({len(PPI)//2} undirected high-conf TF-TF pairs)')

def is_ppi(a, b):
    """Check if a and b are connected by high-conf STRING PPI (symmetric)."""
    return (a, b) in PPI or (b, a) in PPI

# Quick verify
print(f'\nSample PPI checks:')
for a, b in [('GATA1', 'TAL1'), ('PAX5', 'EBF1'), ('RUNX1', 'FLI1'), ('GATA1', 'CD34')]:
    print(f'  is_ppi({a}, {b}) = {is_ppi(a, b)}')

## Part B: Build per-target TF complex partition

For each target gene `t`, find all TFs binding it (incoming TFs from ChIP-DNA prior). Then use STRING PPI to partition these TFs into complexes via **connected-components** on the PPI subgraph induced over incoming TFs.

Note: "connected components" here can be a chain (A-B, B-C → {A,B,C}) which is exactly how yeast's 50-bp peak-distance clustering works — transitive closure of pairwise proximity.

In [ ]:
# Build incoming TFs per target (reverse map of TF_DNA)
incoming_TFs = defaultdict(set)
for source, targets in TF_DNA.items():
    for t in targets:
        incoming_TFs[t].add(source)
incoming_TFs = dict(incoming_TFs)

print(f'Targets with at least one incoming TF: {len(incoming_TFs)} / {n}')
no_incoming = [g for g in Column_order if g not in incoming_TFs]
print(f'Targets with no incoming TF: {len(no_incoming)}')
if no_incoming:
    print(f'  {no_incoming[:10]}')

In [ ]:
# Per-target complex partition via connected-components on PPI subgraph
def find_complexes_on_target(incoming):
    """Given a set of TFs binding a target, partition them into complexes via PPI connectivity.
    Returns a list of TF-sets, each set is one complex.
    Singletons (TFs with no PPI to any other incoming TF) form size-1 complexes.
    """
    incoming = list(incoming)
    # Union-Find via simple DFS
    visited = set()
    complexes = []
    for tf in incoming:
        if tf in visited:
            continue
        # BFS to find all PPI-connected TFs among incoming
        component = []
        stack = [tf]
        while stack:
            cur = stack.pop()
            if cur in visited:
                continue
            visited.add(cur)
            component.append(cur)
            for other in incoming:
                if other not in visited and is_ppi(cur, other):
                    stack.append(other)
        complexes.append(component)
    return complexes

# Build complexes for every target
Complexes_names = {}
for t, in_tfs in incoming_TFs.items():
    complexes = find_complexes_on_target(in_tfs)
    # Keep only non-singleton complexes (size >= 2), like yeast
    Complexes_names[t] = [c for c in complexes if len(c) >= 2]

# Stats
n_targets_with_complex = sum(1 for c in Complexes_names.values() if c)
n_total_complexes = sum(len(c) for c in Complexes_names.values())
complex_sizes = [len(comp) for clist in Complexes_names.values() for comp in clist]

print(f'Targets with at least one TF complex: {n_targets_with_complex} / {n}')
print(f'Total complexes across all targets:   {n_total_complexes}')
if complex_sizes:
    s = pd.Series(complex_sizes)
    print(f'\nComplex sizes:')
    print(f'  mean: {s.mean():.2f}')
    print(f'  max:  {s.max()}')
    print(f'  distribution:')
    print(s.value_counts().sort_index().to_string())

In [ ]:
# Sanity check: show complexes on key textbook targets
print('Complexes on selected textbook targets:\n')
for t in ['HBB', 'HBA1', 'KLF1', 'CD19', 'MPO', 'CSF1R', 'CD14', 'NKG7', 'LEF1', 'TCF4']:
    if t not in Column_order:
        print(f'  {t}: not in panel')
        continue
    in_tfs = sorted(incoming_TFs.get(t, set()))
    complexes = Complexes_names.get(t, [])
    print(f'  {t}:')
    print(f'    Incoming TFs ({len(in_tfs)}): {in_tfs[:12]}{"..." if len(in_tfs)>12 else ""}')
    if complexes:
        for i, comp in enumerate(complexes):
            print(f'    Complex {i+1}: {sorted(comp)}')
    else:
        print(f'    No multi-TF complexes (all TFs are singletons or no PPI among them)')

## Part C: Generate LG_String per source TF (mirror of yeast logic)

For each source TF `s` in Column_order:
- LG_String_temp is a comma-separated list of length 88
- Position `i` value:
  - **Default**: `i` (no constraint; gene at position `i` independent of `s`)
  - **Constrained**: `min(complex_indices)` if `s` is in a complex with TF at position `i` on **some** target

Wait — re-reading yeast code carefully:

```python
for each in Column_order:           # 'each' is the SOURCE TF
    LG_String_temp = ''
    if each not in Complexes_names:  # 'each' is not a target with complexes
        LG_String_temp = ','.join(map(str, range(len(Column_order))))
    else:
        # iterate over complexes ON TARGET 'each'
        for each_complex in Complexes_names[each]:
            ...
```

So yeast's outer loop variable `each` is iterating Column_order as **targets**, not sources. The LG file is indexed by **target**, not source. Each row says "on target `each`, the TFs at these positions are linked into complexes".

Let me follow yeast convention exactly.

In [ ]:
# Build LG_String following yeast convention exactly
# - Outer loop: each gene in Column_order, treated as a TARGET
# - Inner: for that target, build a partition string of length n
#   - default: index i = i  (each position is its own group)
#   - if a TF complex exists on this target, all complex members' indices are
#     remapped to min(complex_indices)
# - Final: ',' + LG_String_temp_1 + ',' + LG_String_temp_2 + ... 
#   then strip the leading ','

name_to_idx = {g: i for i, g in enumerate(Column_order)}

LG_segments = []
for target in Column_order:
    if target not in Complexes_names or not Complexes_names[target]:
        # No complex constraints on this target → default partition
        seg = ','.join(map(str, range(n)))
    else:
        index_converter = {}
        for each_complex in Complexes_names[target]:
            indexes_for_the_complex = [name_to_idx[tf] for tf in each_complex if tf in name_to_idx]
            if len(indexes_for_the_complex) < 2:
                continue
            min_idx = min(indexes_for_the_complex)
            for idx in indexes_for_the_complex:
                index_converter[idx] = min_idx
        # Build segment
        seg_parts = []
        for i in range(n):
            if i in index_converter:
                seg_parts.append(str(index_converter[i]))
            else:
                seg_parts.append(str(i))
        seg = ','.join(seg_parts)
    LG_segments.append(seg)

# Final LG_String: yeast pipeline writes ','.join(segments) effectively
# Yeast code does:  LG_String = LG_String + ',' + LG_String_temp,  then writes LG_String[1:]
# Net result: segments joined by ',' (each segment is itself ','-separated)
# So the total file is one long comma-separated list of n*n numbers
LG_String = ','.join(LG_segments)

n_numbers = LG_String.count(',') + 1
print(f'LG_String total numbers: {n_numbers}  (expected {n*n} = {n}×{n})')
assert n_numbers == n * n, f'LG_String length mismatch: {n_numbers} != {n*n}'

print(f'\nFirst 200 chars of LG_String:')
print(LG_String[:200])
print(f'...\nLast 100 chars:')
print(LG_String[-100:])

In [ ]:
# Show example: HBB segment (a target with strong erythroid complex)
if 'HBB' in name_to_idx:
    hbb_idx = name_to_idx['HBB']
    hbb_seg = LG_segments[hbb_idx]
    hbb_partition = list(map(int, hbb_seg.split(',')))
    
    print(f'HBB (target idx {hbb_idx}) LG segment:')
    print(f'  Default would be: 0,1,2,3,4,...,{n-1}')
    print(f'  Actual:')
    
    # Show only positions where index changed from default (i.e. constrained)
    constrained = [(i, v, Column_order[i]) for i, v in enumerate(hbb_partition) if v != i]
    if constrained:
        print(f'    {len(constrained)} positions constrained:')
        # Group by remapped index
        groups = defaultdict(list)
        for i, remapped, tf in constrained:
            groups[remapped].append(tf)
        # Also include the 'anchor' (the min-index TF that others map to)
        for anchor_idx, tfs in sorted(groups.items()):
            anchor_tf = Column_order[anchor_idx]
            full_complex = sorted({anchor_tf, *tfs})
            print(f'    Complex (anchor idx {anchor_idx}={anchor_tf}): {full_complex}')
    else:
        print('    No constraints (all TFs at default positions)')

## Part D: Save output

In [ ]:
# Save in yeast format: single line, ','-separated, n*n numbers
lg_path = OUTPUT_DIR / 'BMMC_LG.txt'
with open(lg_path, 'w') as f:
    f.write(LG_String)
print(f'Saved: {lg_path}  ({len(LG_String):,} chars, {n_numbers:,} numbers)')

# Also save a row-per-target version for human reading (one segment per line)
lg_rows_path = OUTPUT_DIR / 'BMMC_LG_per_target.tsv'
with open(lg_rows_path, 'w') as f:
    f.write('target_idx\ttarget_gene\tpartition\n')
    for i, seg in enumerate(LG_segments):
        f.write(f'{i}\t{Column_order[i]}\t{seg}\n')
print(f'Saved: {lg_rows_path}  ({n} rows, one per target)')

# Save complex summary (for paper supplementary)
complex_summary_path = OUTPUT_DIR / 'BMMC_LG_complex_summary.tsv'
with open(complex_summary_path, 'w') as f:
    f.write('target_gene\tn_incoming_TFs\tn_complexes\tcomplex_members\n')
    for t in Column_order:
        n_in = len(incoming_TFs.get(t, []))
        complexes = Complexes_names.get(t, [])
        comp_str = ' | '.join(['+'.join(sorted(c)) for c in complexes]) if complexes else '-'
        f.write(f'{t}\t{n_in}\t{len(complexes)}\t{comp_str}\n')
print(f'Saved: {complex_summary_path}')

In [ ]:
# Methods summary
print('=' * 60)
print('Methods summary for paper:')
print('=' * 60)
n_constrained_targets = sum(1 for c in Complexes_names.values() if c)
print(f"""
The SETIA linked-group (LG) constraint file encodes TF complex co-binding
on target gene promoters. Following the yeast pipeline's algorithmic
structure (Rossi et al. ChIP-exo peak proximity at 50 bp), the BMMC adaptation
uses STRING v12.0 high-confidence PPI (combined_score ≥ 900) as the source
of TF co-occupancy evidence in place of ChIP-exo peak distance, since
ChIP-Atlas ChIP-seq peaks lack the basepair resolution required for
distance-based clustering.

For each target gene t, TFs binding t (from the TF-DNA prior) were
partitioned into complexes via connected components on the STRING PPI
subgraph induced over the incoming TFs. Each complex of two or more TFs
was encoded as a shared partition index in the LG file, in yeast-pipeline
format ({n}-position partition string per target, joined into a single
{n}×{n} = {n*n}-element string). A total of {n_constrained_targets} targets
had at least one TF complex; {n_total_complexes} TF complexes were detected
across all targets.
""")